In [1]:
import torch

In [2]:
a=torch.tensor([1.,2.,3.])

In [16]:
def tim(func,inn):
    s=torch.cuda.Event(enable_timing=True)
    e=torch.cuda.Event(enable_timing=True)
    for _ in range(5):
        func(inn)
    s.record()
    func(inn)
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e)

In [10]:
b=torch.randn(100000,10000).cuda()

In [11]:
def s2(a):
    return a*a

In [12]:
def s3(a):
    return a**2

In [17]:
tim(torch.square,b)

14.72976016998291

In [18]:
tim(s2,b)

14.75545597076416

In [20]:
tim(s3,b)

14.722432136535645

In [21]:
with torch.profiler.profile() as prof:
    torch.square(b)

In [23]:
print(prof.key_averages().table(sort_by='cuda_time_total',row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::square         0.15%      29.273us        27.31%       5.471ms       5.471ms       0.000us         0.00%      29.445ms      29.445ms             1  
                                              aten::pow        11.11%       2.225ms        27.17%       5.442ms       5.442ms      14.722ms       100.00%      29.445ms      29.445ms             1  
         

In [24]:
with torch.profiler.profile() as prof:
    s2(b)

In [25]:
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                aten::mul         1.13%     205.594us        20.20%       3.681ms       3.681ms             1  
             Unrecognized        18.53%       3.377ms        18.53%       3.377ms       3.377ms             1  
         cudaLaunchKernel         0.54%      98.425us         0.54%      98.425us      49.213us             2  
    cudaDeviceSynchronize        79.80%      14.544ms        79.80%      14.544ms      14.544ms             1  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
Self CPU time total: 18.225ms



In [26]:
with torch.profiler.profile() as prof:
    s3(b)

In [27]:
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                     Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                aten::pow         0.99%     181.138us        20.01%       3.653ms       3.653ms             1  
        aten::result_type         0.02%       3.953us         0.02%       3.953us       3.953us             1  
                 aten::to         0.01%       1.142us         0.01%       1.142us       1.142us             1  
             Unrecognized        18.60%       3.396ms        18.60%       3.396ms       3.396ms             1  
         cudaLaunchKernel         0.39%      71.405us         0.39%      71.405us      35.703us             2  
    cudaDeviceSynchronize        79.99%      14.601ms        79.99%      14.601ms      14.601ms         